# **PART III — NLP (Spam vs Ham Classification)**

## **1. INSTALL & IMPORT LIBRARIES**

In [ ]:
!pip install gensim
!pip install nltk seaborn gradio wordcloud

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import gensim.downloader as api
import gradio as gr

## **2. DOWNLOAD NLTK RESOURCES**

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

## **3. LOAD DATASET (Google Drive)**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/AI&ML/spamvsham.csv', encoding='latin1')
df.head()

## **4. CHECK & FIX COLUMNS**

In [ ]:
print(df.columns)
print(df.shape)

In [ ]:
# Keep only relevant columns and drop unnamed/NaN-filled extra columns
df = df[['v1', 'v2']]
df = df.rename(columns={'v1': 'label', 'v2': 'text'})
df = df.dropna()
print(df.shape)
df.head()

## **5. LABEL ENCODING**

In [ ]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})
print(df['label'].value_counts())
print(f"Ham: {(df['label']==0).sum()}, Spam: {(df['label']==1).sum()}")

## **6. TEXT CLEANING**

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)      # remove URLs
    text = re.sub(r'@\w+|#\w+', ' ', text)           # remove mentions/hashtags
    text = re.sub(r'\d+', ' ', text)                  # remove numbers
    text = re.sub(r'[^a-zA-Z]', ' ', text)            # remove special characters
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text if word not in stop_words]
    return ' '.join(text)

df['text'] = df['text'].apply(clean_text)
df.head()

## **7. VISUALIZATION — Word Clouds**

In [ ]:
spam_text = ' '.join(df[df['label'] == 1]['text'])
ham_text  = ' '.join(df[df['label'] == 0]['text'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

wc_spam = WordCloud(width=800, height=400, background_color='white',
                    colormap='Reds', max_words=100).generate(spam_text)
axes[0].imshow(wc_spam, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Spam Messages — Word Cloud', fontsize=14, fontweight='bold')

wc_ham = WordCloud(width=800, height=400, background_color='white',
                   colormap='Greens', max_words=100).generate(ham_text)
axes[1].imshow(wc_ham, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Ham Messages — Word Cloud', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## **8. TRAIN-TEST SPLIT**

In [ ]:
X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

## **9. TOKENIZATION**

In [ ]:
# Limit vocabulary to top 10,000 most frequent words
tokenizer = Tokenizer(num_words=10000, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

vocab_size = min(len(tokenizer.word_index) + 1, 10000)
print(f"Vocabulary size: {vocab_size}")

## **10. PADDING**

In [ ]:
# Use 95th percentile of training sequence lengths as MAX_LEN
seq_lengths = [len(seq) for seq in X_train_seq]
max_len = int(np.percentile(seq_lengths, 95))
print(f"MAX_LEN (95th percentile): {max_len}")

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=max_len, padding='post', truncating='post')

print(f"Train shape: {X_train_pad.shape}, Test shape: {X_test_pad.shape}")

## **11. EARLY STOPPING**

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

## **12. MODEL 1 — SIMPLE RNN (BUILD)**

In [ ]:
model_rnn = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64, input_length=max_len),
    SimpleRNN(64),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_rnn.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model_rnn.summary()

## **13. TRAIN RNN**

In [ ]:
history_rnn = model_rnn.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop]
)

## **14. MODEL 2 — LSTM (BUILD)**

In [ ]:
model_lstm = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64, input_length=max_len),
    LSTM(64),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_lstm.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model_lstm.summary()

## **15. TRAIN LSTM**

In [ ]:
history_lstm = model_lstm.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop]
)

## **16. LOAD GLOVE EMBEDDINGS**

In [ ]:
embedding_model = api.load('glove-wiki-gigaword-50')
embedding_dim = 50

## **17. CREATE EMBEDDING MATRIX**

In [ ]:
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if i < vocab_size:
        if word in embedding_model:
            embedding_matrix[i] = embedding_model[word]

print(f"Embedding matrix shape: {embedding_matrix.shape}")

## **18. MODEL 3 — LSTM + GloVe (BUILD)**

In [ ]:
model_w2v = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_len,
        trainable=False
    ),
    LSTM(64),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_w2v.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model_w2v.summary()

## **19. TRAIN WORD2VEC MODEL**

In [ ]:
history_w2v = model_w2v.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop]
)

## **20. TRAINING CURVES**

In [ ]:
def plot_history(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['accuracy'],     label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
    axes[0].set_title(f'{model_name} — Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[1].plot(history.history['loss'],     label='Train Loss')
    axes[1].plot(history.history['val_loss'], label='Val Loss')
    axes[1].set_title(f'{model_name} — Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_history(history_rnn,  'SimpleRNN')
plot_history(history_lstm, 'LSTM')
plot_history(history_w2v,  'LSTM + GloVe')

## **21. EVALUATION FUNCTION**

In [ ]:
def evaluate_model(model, X_test, y_test, name):
    preds = (model.predict(X_test) > 0.5).astype('int32')
    print(f'\n========== {name} ==========')
    print(classification_report(y_test, preds, target_names=['Ham', 'Spam']))
    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Ham', 'Spam'],
                yticklabels=['Ham', 'Spam'])
    plt.title(f'{name} — Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    return accuracy_score(y_test, preds)

## **22. FINAL EVALUATION**

In [ ]:
rnn_acc  = evaluate_model(model_rnn,  X_test_pad, y_test, 'SimpleRNN')
lstm_acc = evaluate_model(model_lstm, X_test_pad, y_test, 'LSTM')
w2v_acc  = evaluate_model(model_w2v,  X_test_pad, y_test, 'LSTM + GloVe')

## **23. ACCURACY COMPARISON BAR CHART**

In [ ]:
models   = ['SimpleRNN', 'LSTM', 'LSTM + GloVe']
accuracy = [rnn_acc, lstm_acc, w2v_acc]

plt.figure(figsize=(8, 5))
bars = plt.bar(models, accuracy, color=['steelblue', 'darkorange', 'green'], edgecolor='black')
plt.ylim(0.9, 1.0)
plt.title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Test Accuracy')

for bar, acc in zip(bars, accuracy):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{acc:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\nSimpleRNN  : {rnn_acc:.4f}')
print(f'LSTM       : {lstm_acc:.4f}')
print(f'LSTM+GloVe : {w2v_acc:.4f}')

## **24. ERROR ANALYSIS**

In [ ]:
# Get predictions from SimpleRNN
rnn_preds = (model_rnn.predict(X_test_pad) > 0.5).astype('int32').flatten()

X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

errors = pd.DataFrame({
    'text':      X_test_reset,
    'true':      y_test_reset,
    'predicted': rnn_preds
})
errors = errors[errors['true'] != errors['predicted']]

print(f'Total errors: {len(errors)}')
print('\n--- FALSE NEGATIVES (Spam predicted as Ham) ---')
fn = errors[errors['true'] == 1]
print(fn[['text', 'true', 'predicted']].head(5).to_string())

print('\n--- FALSE POSITIVES (Ham predicted as Spam) ---')
fp = errors[errors['true'] == 0]
print(fp[['text', 'true', 'predicted']].head(5).to_string())

## **25. REAL-TIME GUI USING GRADIO**

In [ ]:
def predict_message(message):
    cleaned   = clean_text(message)
    sequence  = tokenizer.texts_to_sequences([cleaned])
    padded    = pad_sequences(sequence, maxlen=max_len, padding='post', truncating='post')
    prob      = model_lstm.predict(padded, verbose=0)[0][0]
    label     = 'SPAM' if prob > 0.5 else 'HAM'
    confidence = prob if prob > 0.5 else 1 - prob
    return f'{label}  ({confidence*100:.1f}% confidence)'

demo = gr.Interface(
    fn=predict_message,
    inputs=gr.Textbox(lines=3, placeholder='Type or paste an SMS message here...', label='Input Message'),
    outputs=gr.Textbox(label='Prediction'),
    title='SMS Spam Detector',
    description='Uses a trained LSTM model to classify SMS messages as SPAM or HAM.',
    examples=[
        ['You have won a FREE prize! Call now to claim your reward!'],
        ['Hey are you coming to class tomorrow?'],
        ['Congratulations! You have been selected for a 900 prize reward!'],
        ['Can you pick up some milk on your way home?']
    ]
)

demo.launch(share=True)